In [ ]:
##### model parameters optimization for combined 3 cv situation fs result
### fs list has summarized in R
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFE, VarianceThreshold

from skopt import BayesSearchCV  
from sklearn.metrics import roc_auc_score, f1_score, recall_score, accuracy_score,auc

from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt
import re
import joblib
import ast
from collections import OrderedDict
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, accuracy_score,
    roc_curve, precision_recall_curve, average_precision_score
)

In [ ]:
dat0 = pd.read_csv('../../traindat.csv')
data_x=dat0.drop(['Label'],axis=1)
y=np.array(dat0['Label'])

cat_cols = data_x.select_dtypes(include=['object']).columns
num_cols = data_x.select_dtypes(exclude=['object']).columns

data_x_dum = pd.get_dummies(data_x, columns=cat_cols,dtype=int)
X=data_x_dum

feature_names = [f'feature_{i}' for i in range(X.shape[1])]
feature_to_original = dict(zip(feature_names, X.columns.tolist()))

test_dat = pd.read_csv('../../testdat_withid.csv')
test_ids = test_dat['X'].values
test_x = test_dat.drop(['Label','X'], axis=1)
test_y = np.array(test_dat['Label'])

test_x_dum = pd.get_dummies(test_x, columns=cat_cols, dtype=int)

missing_cols = set(X.columns) - set(test_x_dum.columns)
for col in missing_cols:
    test_x_dum[col] = 0
test_x_dum = test_x_dum[X.columns]  # 保持列顺序一致

selection_df = pd.read_csv('selected_f25.csv')

selection_df['features_list'] = selection_df.iloc[:, 1].apply(
    lambda x: [feat.strip() for feat in re.split(r',|，', x)]  # 支持中英文逗号
)

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示异常

In [ ]:
def get_model(model_name, random_state):
    if model_name == 'rf':
        return RandomForestClassifier(random_state=random_state, 
                                      n_jobs=-1,bootstrap=True,
                                      oob_score=True)
    elif model_name == 'xgb':
        return xgb.XGBClassifier(random_state=random_state)
    elif model_name == 'lgb':
        return lgb.LGBMClassifier(random_state=random_state)
    else:
        raise ValueError(f"未知模型: {model_name}")

param_spaces_optimized = {
   'rf': {
        'n_estimators': (90, 130),   
        'max_depth': (3, 6),        
        'min_samples_leaf': (25, 40), 
        'min_samples_split': (40, 70),
        'max_features': ['sqrt', 0.6],
        'bootstrap': [True],
        'max_samples': (0.6, 0.8),    
        'ccp_alpha': (0.001, 0.01)   
    },
    'xgb': {
        'n_estimators': (50, 75),     
        'max_depth': (2, 4),         
        'learning_rate': (0.045, 0.06),
        'min_child_weight': (3, 6), 
        'subsample': (0.4, 0.55),    
        'colsample_bytree': (0.6, 0.75),
        'gamma': (1.2, 1.8),          
        'reg_alpha': (0.5, 1.2),   
        'reg_lambda': (1.2, 2.0),     
        'scale_pos_weight': (1.5, 3.0)
    },
    'lgb': {
        'n_estimators': (40, 60),    
        'max_depth': (3, 5),         
        'num_leaves': (5, 8),         
        'learning_rate': (0.05, 0.065),
        'min_data_in_leaf': (40, 60), 
        'subsample': (0.45, 0.55),    
        'colsample_bytree': (0.6, 0.7),
        'reg_alpha': (2.2, 3.0),      
        'reg_lambda': (3.5, 4.5),     
        'min_split_gain': (0.2, 0.4), 
        'bagging_freq': (3, 7)        
    }
}

In [ ]:
# save result
result_cols = [
    'mean_train_auc', 'mean_train_f1', 'mean_train_recall', 'mean_train_acc',
    'mean_val_auc', 'mean_val_f1', 'mean_val_recall', 'mean_val_acc',
    'test_auc', 'test_f1', 'test_recall', 'test_acc', 'best_params'
]
for col in result_cols:
    selection_df[col] = None

cv_seed = 888
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed)

date=''

In [ ]:
# with each model
for idx, row in selection_df.iterrows():
    model_name = row.iloc[0] 
    feature_list = row['features_list'] 
    
    try:
        original_features = [feature_to_original[feat] for feat in feature_list if feat in feature_to_original]
        
        if not original_features:
            print(f"索引 {idx}: 未找到有效特征，跳过")
            continue
            
        X_selected = X[original_features].copy()
        test_x_selected = test_x_dum[original_features].copy()
        
        model = get_model(model_name, random_state=cv_seed)
        
        bayes_search = BayesSearchCV(
            estimator=model,
            search_spaces=param_spaces_optimized[model_name], 
            cv=outer_cv,  
            scoring='roc_auc', 
            n_iter=60,  
            random_state=cv_seed,
            n_jobs=-1,
            pre_dispatch='2*n_jobs', 
            n_points=5,
        )
        bayes_search.fit(X_selected, y)
        
        best_params = bayes_search.best_params_
        selection_df.at[idx, 'best_params'] = str(best_params)
        
        train_metrics = {
            'auc': [], 'f1': [], 'recall': [], 'acc': []
        }
        val_metrics = {
            'auc': [], 'f1': [], 'recall': [], 'acc': []
        }
        
        fold_train_roc = [] 
        fold_train_prc = []  
        fold_val_roc = []    
        fold_val_prc = []    

        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_selected, y), 1):
            X_train, X_val = X_selected.iloc[train_idx], X_selected.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            fold_model = get_model(model_name, random_state=cv_seed)
            fold_model.set_params(** best_params)
            
            fold_model.fit(X_train, y_train)
            
            y_train_pred = fold_model.predict(X_train)
            y_train_proba = fold_model.predict_proba(X_train)[:, 1]
            
            y_val_pred = fold_model.predict(X_val)
            y_val_proba = fold_model.predict_proba(X_val)[:, 1]
            
            train_metrics['auc'].append(roc_auc_score(y_train, y_train_proba))
            train_metrics['f1'].append(f1_score(y_train, y_train_pred))
            train_metrics['recall'].append(recall_score(y_train, y_train_pred))
            train_metrics['acc'].append(accuracy_score(y_train, y_train_pred))
            
            val_metrics['auc'].append(roc_auc_score(y_val, y_val_proba))
            val_metrics['f1'].append(f1_score(y_val, y_val_pred))
            val_metrics['recall'].append(recall_score(y_val, y_val_pred))
            val_metrics['acc'].append(accuracy_score(y_val, y_val_pred))
        
        selection_df.at[idx, 'mean_train_auc'] = np.mean(train_metrics['auc'])
        selection_df.at[idx, 'mean_train_f1'] = np.mean(train_metrics['f1'])
        selection_df.at[idx, 'mean_train_recall'] = np.mean(train_metrics['recall'])
        selection_df.at[idx, 'mean_train_acc'] = np.mean(train_metrics['acc'])
        
        selection_df.at[idx, 'mean_val_auc'] = np.mean(val_metrics['auc'])
        selection_df.at[idx, 'mean_val_f1'] = np.mean(val_metrics['f1'])
        selection_df.at[idx, 'mean_val_recall'] = np.mean(val_metrics['recall'])
        selection_df.at[idx, 'mean_val_acc'] = np.mean(val_metrics['acc'])
        
        final_model = get_model(model_name, random_state=cv_seed)
        final_model.set_params(**best_params)
        final_model.fit(X_selected, y)
        
        if y is not None:
            y_pred = final_model.predict(X_selected)
            y_proba = final_model.predict_proba(X_selected)[:, 1]
        
            selection_df.at[idx, 'train_auc'] = roc_auc_score(y, y_proba)
            selection_df.at[idx, 'train_f1'] = f1_score(y, y_pred)
            selection_df.at[idx, 'train_recall'] = recall_score(y, y_pred)
            selection_df.at[idx, 'train_acc'] = accuracy_score(y, y_pred)

            trainall_auc = roc_auc_score(y, y_proba)
            trainall_fpr, trainall_tpr, _ = roc_curve(y, y_proba)
        else:
            selection_df.at[idx, 'train_auc'] = np.nan
            selection_df.at[idx, 'train_f1'] = np.nan
            selection_df.at[idx, 'train_recall'] = np.nan
            selection_df.at[idx, 'train_acc'] = np.nan
            print("未提供测试集，跳过测试集评估")
        
        if test_y is not None:
            y_test_pred = final_model.predict(test_x_selected)
            y_test_proba = final_model.predict_proba(test_x_selected)[:, 1]
        
            selection_df.at[idx, 'test_auc'] = roc_auc_score(test_y, y_test_proba)
            selection_df.at[idx, 'test_f1'] = f1_score(test_y, y_test_pred)
            selection_df.at[idx, 'test_recall'] = recall_score(test_y, y_test_pred)
            selection_df.at[idx, 'test_acc'] = accuracy_score(test_y, y_test_pred)
            
            test_auc = roc_auc_score(test_y, y_test_proba)
            test_fpr, test_tpr, _ = roc_curve(test_y, y_test_proba)
        else:
            selection_df.at[idx, 'test_auc'] = np.nan
            selection_df.at[idx, 'test_f1'] = np.nan
            selection_df.at[idx, 'test_recall'] = np.nan
            selection_df.at[idx, 'test_acc'] = np.nan
            print("未提供测试集，跳过测试集评估")
    except Exception as e:
        print(f"索引 {idx} 处理出错: {str(e)}")
        continue

In [ ]:
#### fitting final model and auc plot and shap
import shap 
from collections import defaultdict
# 中文显示设置
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.ioff() 

In [ ]:
cv_seed = 888
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed)
save_path = "./results/"  

# 定义模型
def get_model(model_name, random_state):
    if model_name == 'rf':
        return RandomForestClassifier(random_state=random_state, 
                                      n_jobs=-1,bootstrap=True,
                                      max_depth=4,
                                      max_features='log2', 
                                      min_samples_leaf=10, 
                                      min_samples_split=10,
                                      max_samples = 0.72,
                                      n_estimators=180,
                                      oob_score=True)
    elif model_name == 'xgb':
        return xgb.XGBClassifier(random_state=random_state,
                                 colsample_bytree= 0.8,
                                 gamma=1.0977281479105039,
                                 learning_rate=0.06218323339845347, 
                                 max_depth=4,
                                 min_child_weight=2, 
                                 n_estimators=75,
                                 reg_alpha= 0.4224451533220679,
                                 reg_lambda=0.7, 
                                 subsample=0.4336198686662374)
    elif model_name == 'lgb':
        return lgb.LGBMClassifier(random_state=random_state,
                                  colsample_bytree= 0.6015359515446118,
                                  learning_rate=0.055,
                                  max_depth=4,
                                  min_data_in_leaf=33,
                                  min_split_gain=0.11, 
                                  n_estimators=60,
                                  num_leaves=5,
                                  reg_alpha=1.7, 
                                  reg_lambda=2.9,
                                  subsample=0.633305319859095)
    else:
        raise ValueError(f"未知模型: {model_name}")

fold_auc_results = []

model_performance = []

test_pred_dict = {} 

In [ ]:
for idx, row in selection_df.iterrows():
    model_name = row.iloc[0]
    feature_list = row['features_list']
    
    try:
        original_features = [feature_to_original[feat] for feat in feature_list if feat in feature_to_original]
        if not original_features:
            print(f"索引 {idx}: 未找到有效特征，跳过")
            continue
        
        X_selected = X[original_features].copy()
        test_x_selected = test_x_dum[original_features].copy()
        test_y_true = test_y
        
        # 初始化模型
        model = get_model(model_name,random_state=cv_seed)
        
        cv_fprs = []
        cv_tprs = []
        cv_aucs = []
        train_aucs = [] 
        val_aucs = []  
        val_all_y = []    
        val_all_proba = []

        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
        plt.figure(figsize=(8, 6)) 
        
        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_selected, y), 1):
            X_train, X_val = X_selected.iloc[train_idx], X_selected.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            model.fit(X_train, y_train)
            
            train_proba = model.predict_proba(X_train)[:, 1]
            train_auc = roc_auc_score(y_train, train_proba)
            train_aucs.append(train_auc)
            
            val_proba = model.predict_proba(X_val)[:, 1]
            val_auc = roc_auc_score(y_val, val_proba)
            val_aucs.append(val_auc)

            val_all_y.extend(y_val)
            val_all_proba.extend(val_proba)
            
            val_fpr, val_tpr, _ = roc_curve(y_val, val_proba)
            cv_fprs.append(val_fpr)
            cv_tprs.append(val_tpr)
            cv_aucs.append(val_auc)
            
            plt.plot(val_fpr, val_tpr, lw=1.5, alpha=0.7,color=colors[fold-1],
                     label=f'Fold {fold} (AUC = {val_auc:.3f})')
            
            fold_auc_results.append({
                'model_name': model_name,
                'fold': fold,
                'train_auc': train_auc,
                'val_auc': val_auc
            })
            
            print(f"模型 {model_name} 折 {fold} - 训练集AUC: {train_auc:.4f}, 验证集AUC: {val_auc:.4f}")
        
        mean_val_auc = np.mean(val_aucs)
        std_val_auc = np.std(val_aucs)
        mean_cv_auc = np.mean(cv_aucs)
        
        plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'{model_name} ROC plot of 5-fold validation set')
        plt.legend(loc='lower right', fontsize='small')
        plt.tight_layout()
        plt.savefig(f"{save_path}{model_name}_cv_roc_tiled.png", dpi=300, bbox_inches='tight')
        plt.close()

        val_all_fpr, val_all_tpr, _ = roc_curve(val_all_y, val_all_proba)
        val_all_auc = auc(val_all_fpr, val_all_tpr)
        
        final_model = get_model(model_name,random_state=cv_seed)
        final_model.fit(X_selected, y)
        
        train_full_proba = final_model.predict_proba(X_selected)[:, 1]
        train_full_auc = roc_auc_score(y, train_full_proba)
        train_full_fpr, train_full_tpr, _ = roc_curve(y, train_full_proba)
        
        test_proba = final_model.predict_proba(test_x_selected)[:, 1]
        test_pred = final_model.predict(test_x_selected)
        test_auc = roc_auc_score(test_y_true, test_proba)
        test_fpr, test_tpr, _ = roc_curve(test_y_true, test_proba)
        
        test_pred_df = pd.DataFrame({
            'id': test_ids,
            'original_label': test_y_true,
            'pred_prob': test_proba,
            'pred_label': test_pred
        })
        test_pred_dict[model_name] = test_pred_df  # 存入字典
        test_pred_df.to_csv(f"{save_path}{model_name}_test_pred_results.csv", index=False, encoding='utf-8-sig')
        test_pred_results.append(test_pred_df)

        plt.figure(figsize=(8, 6))
        plt.plot(val_all_fpr, val_all_tpr, color='green', lw=2,
                 label=f'合并验证集 (AUC = {val_all_auc:.3f}, 均值±SD = {mean_val_auc:.3f} ± {std_val_auc:.3f})')
        plt.plot(train_full_fpr, train_full_tpr, color='blue', lw=2,
                 label=f'全训练集 (AUC = {train_full_auc:.3f})')
        plt.plot(test_fpr, test_tpr, color='red', lw=2,
                 label=f'测试集 (AUC = {test_auc:.3f})')
        # 对角线
        plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'{model_name} 模型ROC曲线（合并验证集+训练集+测试集）')
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.savefig(f"{save_path}{model_name}_merged_val_train_test_roc.png", dpi=300, bbox_inches='tight')
        plt.close()
        
        model_performance.append({
            'model_name': model_name,
            'mean_train_auc': np.mean(train_aucs),
            'std_train_auc': np.std(train_aucs),
            'mean_val_auc': mean_val_auc,
            'std_val_auc': std_val_auc,
            'merged_val_auc': val_all_auc,
            'train_full_auc': train_full_auc,
            'test_auc': test_auc
        })
        
        print(f"开始 {model_name} 模型SHAP分析...")
        if model_name == "rf":
            explainer = shap.TreeExplainer(final_model)
            shap_values = explainer.shap_values(X_selected)
            if len(shap_values.shape) == 3:  
                shap_values = shap_values[:, :, 1]  
        elif model_name in ["xgb", "lgb"]:
            explainer = shap.TreeExplainer(final_model)
            shap_values = explainer.shap_values(X_selected)
            if isinstance(shap_values, list) and len(shap_values) == 2:
                shap_values = shap_values[1]

        if shap_values.shape[1] != X_selected.shape[1]:
            raise ValueError(f"SHAP值维度({shap_values.shape})与特征维度({X_selected.shape})不匹配")

        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_selected, show=False)
        plt.title(f"{model_name} 模型SHAP特征重要性")
        plt.tight_layout()
        plt.savefig(f"{save_path}{model_name}_shap_summary.png", dpi=300, bbox_inches='tight')
        plt.close()
        
        top_features = X_selected.columns[:2]
        for feat in top_features:
            plt.figure(figsize=(8, 5))
            shap.dependence_plot(feat, shap_values, X_selected, show=False)
            plt.title(f"{model_name} 模型{feat} SHAP依赖图")
            plt.tight_layout()
            plt.savefig(f"{save_path}{model_name}_shap_dependence_{feat}.png", dpi=300, bbox_inches='tight')
            plt.close()
        
    except Exception as e:
        print(f"处理模型 {model_name} 时出错: {str(e)}")
        continue

fold_auc_df = pd.DataFrame(fold_auc_results)
fold_auc_df.to_csv(f"{save_path}fold_wise_auc_results.csv", index=False, encoding='utf-8-sig')

model_perf_df = pd.DataFrame(model_performance)
model_perf_df.to_csv(f"{save_path}model_overall_performance.csv", index=False, encoding='utf-8-sig')

if test_pred_dict:
    all_test_pred_df = pd.concat(test_pred_dict.values(), keys=test_pred_dict.keys(), names=['model_name'])
    all_test_pred_df.to_csv(f"{save_path}all_models_test_pred_results.csv", index=True, encoding='utf-8-sig')

In [ ]:
# 绘制特定几个特征的特征依赖图
# XGB
#idx=1
#row=selection_df.iloc[1]
# RF
#idx=0
#row=selection_df.iloc[0]
# LGBM
idx=2
row=selection_df.iloc[2]


model_name = row.iloc[0]
feature_list = row['features_list']

original_features = [feature_to_original[feat] for feat in feature_list if feat in feature_to_original]
if not original_features:
    print(f"索引 {idx}: 未找到有效特征，跳过")

X_selected0 = X[original_features].copy()

In [ ]:
original_to_display_mapping = {
    "MET109": "HPHPA",
    "MET148": "Cholic acid",
    "MET189": "Indolelactic acid",
    "Age": "Age",
    "BMI": "BMI",
    "Economic.conditions": "Economic conditions",
    "Education_tertiary": "Education_level_tertiary",
    "Marriage_single": "Marriage_single",
    "Pre.Work_full-time": "Pre-Work_full-time",
    "MET28": "Protocatechuic acid",
    "MET41": "o-HPAA",
    "MET42": "Hippuric acid",
    "MET53": "Salicyluric acid", 
    "MET67": "Glycine", 
    "MET72": "Glycolic acid",
    "MET92": "2-HIBA",
    "MET193": "GCDCA-3S",
    "Education_secondary": "Education_level_secondary",
    "Pre.Residence_城市": "Pre-Residence_urban",
    "Pre.Income":"Pre-Income"
}

existing_features = [feat for feat in original_features if feat in X_selected0.columns]
display_features = [original_to_display_mapping.get(feat, feat) for feat in existing_features]

X_selected = X_selected0[existing_features].rename(
    columns=dict(zip(existing_features, display_features))
)

In [ ]:
final_model = get_model(model_name,random_state=cv_seed)
final_model.fit(X_selected, y)

print(f"开始 {model_name} 模型SHAP分析...")
if model_name == "rf":
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_selected)
    if len(shap_values.shape) == 3:  
        shap_values = shap_values[:, :, 1] 
elif model_name in ["xgb", "lgb"]:
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_selected)
    if isinstance(shap_values, list) and len(shap_values) == 2:
        shap_values = shap_values[1]

if shap_values.shape[1] != X_selected.shape[1]:
    raise ValueError(f"SHAP值维度({shap_values.shape})与特征维度({X_selected.shape})不匹配")

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_selected, show=False)
plt.title("")
plt.tight_layout()
plt.savefig(f"{save_path}{model_name}_shap_summary_tg.png", dpi=300, bbox_inches='tight')
plt.close()

top_features = ["Salicyluric acid", "HPHPA", "Glycolic acid", "Glycine", "Protocatechuic acid", "o-HPAA"]
for feat in top_features:
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(feat, shap_values, X_selected, show=False,interaction_index=None)
    plt.title("")
    plt.xlabel('Concentration')
    plt.tight_layout()
    plt.savefig(f"{save_path}{model_name}_shap_dependence_{feat}_noninteraction_tg.png", dpi=300, bbox_inches='tight')
    plt.close()